In [ ]:
import numpy as np
import pynumdiff

from pynumdiff.utils import simulate, evaluate


# Generate testing data

Here we amplify the usual signal to get outside the -pi to pi bound.

In [ ]:
noise_type = 'normal' # noise is generated using np.random, e.g. 'normal', 'uniform', 'poisson'
noise_parameters = [0, 0.5]  # compatible with np.random functions 
random_seed = 1

dt = 0.01 # step size and series length in terms of independent variable
duration = 4

x, x_truth, dxdt_truth = simulate.lorenz_x(duration=duration, dt=dt, outliers=False,
                                           noise_type=noise_type, noise_parameters=noise_parameters)

# amplify signal
gain = 4
x_truth *= gain
dxdt_truth *= gain

# add noise
x = simulate._add_noise(x_truth, random_seed, noise_type, noise_parameters)

# wrap to [-pi, pi]
x = (x + np.pi) % (2*np.pi) - np.pi
x_truth = (x_truth + np.pi) % (2*np.pi) - np.pi


# Naive numerical differentiation (without considering wrapping)

In [ ]:
x_hat, dxdt_hat = pynumdiff.kalman_smooth.rtsdiff(x, dt, 1, 5, True, axis=0, circular=False)
x_hat_wrapped = (x_hat + np.pi) % (2*np.pi) - np.pi

evaluate.plot(x, dt, x_hat_wrapped, dxdt_hat, x_truth, dxdt_truth)


# Now with circular=True


In [ ]:
x_hat, dxdt_hat = pynumdiff.kalman_smooth.rtsdiff(x, dt, 1, 3, True, axis=0, circular=True)
x_hat_wrapped = (x_hat + np.pi) % (2*np.pi) - np.pi

evaluate.plot(x, dt, x_hat_wrapped, dxdt_hat, x_truth, dxdt_truth)


# Test multidimensional

In [ ]:
v, v_truth, dvdt_truth = simulate.triangle(duration=duration, dt=dt, outliers=False,
                                           noise_type=noise_type, noise_parameters=noise_parameters)


In [ ]:
X = np.vstack((x, v)).T
print('Shape:', X.shape)


In [ ]:
# Differentiate circular and non-circular dimensions separately
x_hat_col, dxdt_hat_col = pynumdiff.kalman_smooth.rtsdiff(X[:,0], dt, 1, 3, True, circular=True)
v_hat_col, dvdt_hat_col = pynumdiff.kalman_smooth.rtsdiff(X[:,1], dt, 1, 3, True)

x_hat_wrapped = (x_hat_col + np.pi) % (2*np.pi) - np.pi
evaluate.plot(x, dt, x_hat_wrapped, dxdt_hat_col, x_truth, dxdt_truth)
